To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://unsloth.ai/docs/get-started/install).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

!uv pip install --upgrade wandb protobuf

In [ ]:
!pip install torch==2.5.1
!pip install transformers==4.46.0
!pip install accelerate==1.2.1
!pip install datasets==3.1.0
!pip install safetensors==0.4.5
!pip install trl==0.17.0
!pip install vllm==0.11.2
!pip install unsloth


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:False"
os.environ["UNSLOTH_VLLM_STANDBY"] = "0"


In [ ]:
import os
import re
import inspect
from typing import List, Sequence

from datasets import load_dataset
from safetensors import safe_open
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel
from vllm import SamplingParams

import torch


# -----------------------------------------------------------------------------
# Model and LoRA configuration
# -----------------------------------------------------------------------------

max_seq_length = 2048  # Qwen3 supports longer context
lora_rank = 32  # Larger rank = better capacity, but slower / more memory.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # False for LoRA 16bit training.
    load_in_fp8 = False,
    fast_inference=True,  # Enable vLLM-backed fast inference.
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # Reduce if you hit OOM.
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,  # Suggested values: 8, 16, 32, 64, 128.
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Remove some modules (e.g. QKVO) if out of memory.
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",  # Helps with long-context training.
    random_state=3407,
)


# -----------------------------------------------------------------------------
# Dataset preparation (GSM8K) and prompt format
# -----------------------------------------------------------------------------

dataset = load_dataset("openai/gsm8k", "main", split="train")
print(dataset)
print(dataset[0]["question"])
print(dataset[0]["answer"])


def extract_hash_answer(text: str) -> str | None:
    """Extract the ground-truth numeric/string answer after '####' from GSM8K."""
    if "####" not in text:
        return None
    return text.split("####", maxsplit=1)[1].strip()


print(extract_hash_answer(dataset[0]["answer"]))


def estimate_difficulty(answer: str) -> int:
    """
    Estimate difficulty based on the number of operations in the answer.
    GSM8K problems typically require 2-8 steps to solve.

    Simple (2-4 steps): difficulty=0
    Medium (5-6 steps): difficulty=1
    Hard (7+ steps): difficulty=2
    """
    if answer is None:
        return 1

    # Count operators that indicate multi-step reasoning
    operators = ['+', '-', '*', '/', '×', '÷', '=', 'of', 'per', 'each', 'times']
    count = sum(1 for op in operators if op in answer.lower())

    # Also consider the length as a secondary factor
    words = len(answer.split())

    # Estimate: more operators and longer answers = more steps = harder
    estimated_steps = max(count, words // 3)

    if estimated_steps <= 4:
        return 0  # Easy
    elif estimated_steps <= 6:
        return 1  # Medium
    else:
        return 2  # Hard


# Show difficulty distribution
print("\nEstimating difficulty for first 10 examples:")
for i in range(10):
    diff = estimate_difficulty(dataset[i]["answer"])
    print(f"Example {i}: difficulty={diff}, answer={dataset[i]['answer']}")

reasoning_start = "<start_working_out>"
reasoning_end = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = (
    "You are a helpful math tutor. Solve the following problem step by step.\n"
    "First, provide your reasoning between <start_working_out> and <end_working_out>.\n"
    "Then, provide your final answer between <SOLUTION> and </SOLUTION>.\n"
    "Example:\n"
    "<start_working_out>\n"
    "Let me solve this step by step...\n"
    "Step 1: ...\n"
    "Step 2: ...\n"
    "<end_working_out>\n"
    "<SOLUTION>42</SOLUTION>"
)
print(system_prompt)

# Map raw GSM8K entries into chat-style prompts + extracted answers.
dataset = dataset.map(
    lambda x: {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": x["question"]},
        ],
        "answer": extract_hash_answer(x["answer"]),
        "difficulty": estimate_difficulty(x["answer"]),
    }
)
print(dataset[0])

# Show difficulty distribution
print("\nDifficulty distribution:")
difficulty_counts = {0: 0, 1: 0, 2: 0}
for example in dataset:
    difficulty_counts[example["difficulty"]] += 1
print(f"Easy (0): {difficulty_counts[0]} ({100*difficulty_counts[0]/len(dataset):.1f}%)")
print(f"Medium (1): {difficulty_counts[1]} ({100*difficulty_counts[1]/len(dataset):.1f}%)")
print(f"Hard (2): {difficulty_counts[2]} ({100*difficulty_counts[2]/len(dataset):.1f}%)")


# -----------------------------------------------------------------------------
# Reward functions: enforce format + numeric correctness
# -----------------------------------------------------------------------------

match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{reasoning_start}.+?{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL,
)

# Quick sanity-check of the regex.
match_format.search(
    "<start_working_out>Let me think!<end_working_out>"
    "<SOLUTION>2</SOLUTION>",
)


def match_format_exactly(completions, **kwargs) -> List[float]:
    """Reward only if the full expected format is matched exactly."""
    scores: List[float] = []
    for completion in completions:
        score = 0.0
        response = completion[0]["content"]
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores


def match_format_approximately(completions, **kwargs) -> List[float]:
    """Soft reward based on how many times the special tags appear."""
    scores: List[float] = []
    for completion in completions:
        score = 0.0
        response = completion[0]["content"]
        # We want each tag to appear exactly once; more or fewer is penalized.
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        score += 0.5 if response.count(solution_start) == 1 else -1.0
        score += 0.5 if response.count(solution_end) == 1 else -1.0
        scores.append(score)
    return scores


def check_answer(prompts, completions, answer, **kwargs) -> List[float]:
    """Reward based on how close the extracted final answer is to the ground truth."""
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]

    scores: List[float] = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0.0
        if guess is None:
            scores.append(0.0)
            continue

        # Exact string match gets the highest reward.
        if guess == true_answer:
            score += 3.0
        # Same up to whitespace differences.
        elif guess.strip() == (true_answer or "").strip():
            score += 1.5
        else:
            # Allow partial numeric credit if the ratio is close.
            try:
                guess_f = float(guess)
                true_f = float(true_answer)
                if 0.9 <= guess_f / true_f <= 1.1:
                    score += 1.0
                elif 0.8 <= guess_f / true_f <= 1.2:
                    score += 0.5
                else:
                    score -= 1.5  # Penalize wrong answers.
            except Exception:
                score -= 1.5  # Penalize non-numeric / unparsable answers.
        scores.append(score)
    return scores


match_numbers = re.compile(
    solution_start + r".*?([\d\.\,]{1,})",
    flags=re.MULTILINE | re.DOTALL,
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))

PRINTED_TIMES = 0
PRINT_EVERY_STEPS = 5


def check_numbers(prompts, completions, answer, **kwargs) -> List[float]:
    """Reward if the numeric value inside <SOLUTION> matches the ground truth."""
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1) if (guess := match_numbers.search(r)) is not None else None
        for r in responses
    ]

    scores: List[float] = []
    global PRINTED_TIMES, PRINT_EVERY_STEPS

    # Light-weight logging for debugging the learned behavior.
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            "*" * 20,
            f"Question:\n{question}",
            f"\nAnswer:\n{answer[0]}",
            f"\nResponse:\n{responses[0]}",
            f"\nExtracted:\n{extracted_responses[0]}",
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(0.0)
            continue

        try:
            true_f = float((true_answer or "").strip())
            # Remove commas like in "123,456".
            guess_f = float(guess.strip().replace(",", ""))
            scores.append(1.5 if guess_f == true_f else -0.5)
        except Exception:
            scores.append(0.0)
    return scores


# -----------------------------------------------------------------------------
# Training configuration (GRPO)
# -----------------------------------------------------------------------------

# Curriculum Learning configuration
max_prompt_length = 512 + 1  # +1 as a small safety margin for Qwen3.

# Curriculum learning: dynamically adjust difficulty based on training progress
def get_allowed_difficulties(step: int, max_steps: int = 500) -> List[int]:
    """
    Return allowed difficulty levels based on current training step.

    Stage 1 (0-30%): Easy + Medium only
    Stage 2 (30-70%): All difficulties
    Stage 3 (70-100%): Medium + Hard (review)
    """
    progress = step / max_steps

    if progress < 0.3:
        return [0, 1]  # Easy + Medium
    elif progress < 0.7:
        return [0, 1, 2]  # All difficulties
    else:
        return [1, 2]  # Medium + Hard


# Create filtered datasets for each difficulty level
dataset_by_difficulty = {
    0: dataset.filter(lambda x: x["difficulty"] == 0),
    1: dataset.filter(lambda x: x["difficulty"] == 1),
    2: dataset.filter(lambda x: x["difficulty"] == 2),
}

print(f"\nDataset sizes by difficulty:")
print(f"Easy (0): {len(dataset_by_difficulty[0])}")
print(f"Medium (1): {len(dataset_by_difficulty[1])}")
print(f"Hard (2): {len(dataset_by_difficulty[2])}")

# Create curriculum dataset that combines all difficulties with balanced sampling
from datasets import concatenate_datasets, Dataset as HFDataset


def get_curriculum_dataset(allowed_difficulties: List[int]) -> "HFDataset":
    """Get dataset with difficulty distribution based on allowed difficulties."""
    # Sample equally from each allowed difficulty
    datasets_to_concat = []
    min_size = min(len(dataset_by_difficulty[d]) for d in allowed_difficulties)

    for d in allowed_difficulties:
        # Sample the same number from each difficulty for balance
        subset = dataset_by_difficulty[d].shuffle(seed=3407 + d).select(range(min_size))
        datasets_to_concat.append(subset)

    combined = concatenate_datasets(datasets_to_concat)
    return combined.shuffle(seed=3407)


# Use full dataset with curriculum learning via callback
# The trainer will iterate through the dataset, and we'll track progress
full_dataset = get_curriculum_dataset([0, 1, 2])

training_args = GRPOConfig(
    learning_rate=5e-6,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Increase for smoother updates if you have time.
    num_generations=4,  # Decrease if you hit OOM.
    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,
    # num_train_epochs=1,  # Alternative to max_steps for full-epoch training.
    max_steps=200,
    save_steps=100,
    max_grad_norm=1.0,
    report_to="none",  # Integrate with Weights & Biases etc. if needed.
    output_dir="outputs",
)


# Curriculum learning: Create a callback to update dataset based on training progress
from transformers import TrainerCallback
from datasets import Dataset


class CurriculumLearningCallback(TrainerCallback):
    """
    Callback that updates the training dataset based on curriculum difficulty.
    This implements the three-stage curriculum:
    - Stage 1 (0-30%): Easy + Medium
    - Stage 2 (30-70%): All difficulties
    - Stage 3 (70-100%): Medium + Hard
    """

    def __init__(self, trainer, dataset_by_difficulty, get_allowed_difficulties_fn):
        self.trainer = trainer
        self.dataset_by_difficulty = dataset_by_difficulty
        self.get_allowed_difficulties = get_allowed_difficulties_fn
        self.last_update_step = -1

    def on_step_begin(self, args, state, control, **kwargs):
        """Called at the beginning of each training step."""
        current_step = state.global_step
        max_steps = args.max_steps
        allowed_difficulties = self.get_allowed_difficulties(current_step, max_steps)

        # Only update when moving to a new curriculum stage
        current_stage = self._get_stage(current_step, max_steps)
        last_stage = self._get_stage(self.last_update_step, max_steps) if self.last_update_step >= 0 else -1

        if current_stage != last_stage:
            print(f"\n[Curriculum] Step {current_step}/{max_steps}: Switching to stage {current_stage}")
            print(f"[Curriculum] Allowed difficulties: {allowed_difficulties}")

            # Create new balanced dataset
            new_dataset = self._create_balanced_dataset(allowed_difficulties)
            self.trainer.train_dataset = new_dataset
            self.last_update_step = current_step

    def _get_stage(self, step, max_steps):
        """Get curriculum stage for a given step."""
        progress = step / max_steps
        if progress < 0.3:
            return 1
        elif progress < 0.7:
            return 2
        else:
            return 3

    def _create_balanced_dataset(self, allowed_difficulties):
        """Create a balanced dataset from allowed difficulty levels."""
        from datasets import concatenate_datasets

        # Sample equally from each difficulty
        min_size = min(len(self.dataset_by_difficulty[d]) for d in allowed_difficulties)
        datasets_to_concat = []

        for d in allowed_difficulties:
            subset = self.dataset_by_difficulty[d].shuffle(seed=3407 + d).select(range(min_size))
            datasets_to_concat.append(subset)

        combined = concatenate_datasets(datasets_to_concat)
        return combined.shuffle(seed=3407)


# Create curriculum callback
curriculum_callback = CurriculumLearningCallback(
    trainer=None,  # Will be set after trainer creation
    dataset_by_difficulty=dataset_by_difficulty,
    get_allowed_difficulties_fn=get_allowed_difficulties,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args=training_args,
    train_dataset=full_dataset,
    callbacks=[curriculum_callback],
)

# Set trainer reference in callback after creation
curriculum_callback.trainer = trainer

trainer.train()




In [ ]:
model.save_lora("grpo_saved_lora")

# Basic sanity-check that LoRA weights are non-trivial (not all zeros).
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework="pt") as f:
    for key in f.keys():
        tensor = f.get_tensor(key)
        zero_fraction = (tensor == 0).sum().item() / tensor.numel()
        assert zero_fraction < 1.0, f"LoRA tensor {key} appears to be all zeros."


sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)


def _single_completion(
    model,
    tokenizer,
    messages: Sequence[dict],
    sampling_params: SamplingParams,
    lora_request=None,
) -> str:
    """Helper: generate a single completion given chat messages."""
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    outputs = model.fast_generate(
        [text],
        sampling_params=sampling_params,
        lora_request=lora_request,
    )
    return outputs[0].outputs[0].text


def evaluate_base_vs_lora_on_dataset(
    model,
    tokenizer,
    sampling_params: SamplingParams,
    eval_dataset,
    lora_path: str,
    num_examples: int = 5,
) -> None:
    """Compare base model vs. trained LoRA on several GSM8K-style questions."""
    lora_request = model.load_lora(lora_path)
    reward_fns = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ]

    n = min(num_examples, len(eval_dataset))
    total_base = [0.0 for _ in reward_fns]
    total_lora = [0.0 for _ in reward_fns]

    for idx in range(n):
        sample = eval_dataset[idx]
        prompts = sample["prompt"]
        answers = [sample["answer"]]

        # TRL reward functions expect prompts in format: [[messages]]
        prompts_for_reward = [prompts]

        base_output = _single_completion(
            model,
            tokenizer,
            prompts,
            sampling_params,
            lora_request=None,
        )
        lora_output = _single_completion(
            model,
            tokenizer,
            prompts,
            sampling_params,
            lora_request=lora_request,
        )

        base_completions = [[{"role": "assistant", "content": base_output}]]
        lora_completions = [[{"role": "assistant", "content": lora_output}]]

        # 根据函数签名使用正确的参数顺序
        base_scores = []
        lora_scores = []
        for fn in reward_fns:
            sig = inspect.signature(fn)
            params = list(sig.parameters.keys())

            if params[0] == "prompts":
                # check_answer, check_numbers: (prompts, completions, answer)
                base_scores.append(fn(prompts_for_reward, base_completions, answers)[0])
                lora_scores.append(fn(prompts_for_reward, lora_completions, answers)[0])
            else:
                # match_format_*: (completions, prompts=..., answer=...)
                base_scores.append(fn(base_completions, prompts=prompts_for_reward, answer=answers)[0])
                lora_scores.append(fn(lora_completions, prompts=prompts_for_reward, answer=answers)[0])

        total_base = [tb + sb for tb, sb in zip(total_base, base_scores)]
        total_lora = [tl + sl for tl, sl in zip(total_lora, lora_scores)]

        print("-" * 80)
        print(f"Example {idx}")
        print(f"Question:\n{prompts[-1]['content']}")
        print(f"True answer:\n{answers[0]}")
        print(f"\nBase output:\n{base_output}")
        print(f"\nLoRA output:\n{lora_output}")
        print(f"Base rewards: {base_scores}")
        print(f"LoRA rewards: {lora_scores}")
        print(f"Base rewards: {base_scores}")
        print(f"LoRA rewards: {lora_scores}")

    print("=" * 80)
    print(f"Averaged over {n} examples:")
    print("Base rewards:", [round(v / n, 3) for v in total_base])
    print("LoRA rewards:", [round(v / n, 3) for v in total_lora])


# Use a small subset of the training set as a quick sanity-check.
eval_subset = dataset.select(range(5))
evaluate_base_vs_lora_on_dataset(
    model=model,
    tokenizer=tokenizer,
    sampling_params=sampling_params,
    eval_dataset=eval_subset,
    lora_path="grpo_saved_lora",
    num_examples=5,
)

# Additionally, show a single hand-crafted comparison (sqrt(101)) for intuition.
custom_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the sqrt of 101?"},
]

base_answer = _single_completion(
    model,
    tokenizer,
    custom_messages,
    sampling_params,
    lora_request=None,
)
fine_tuned_answer = _single_completion(
    model,
    tokenizer,
    custom_messages,
    sampling_params,
    lora_request=model.load_lora("grpo_saved_lora"),
)

print("-" * 80)
print("Custom question: sqrt(101)")
print("Base model output:\n", base_answer)
print("LoRA fine-tuned output:\n", fine_tuned_answer)

### Unsloth

Load up `Llama 3.2 3B Instruct`, and set parameters. To finetune a base model from scratch, check out our `Qwen 3 4B Base GRPO` notebook [here](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

### Data Prep
<a name="Data"></a>

We're using OpenAI's famous GSM8K dataset!

In [ ]:
from datasets import load_dataset
dataset = load_dataset("openai/gsm8k", "main", split = "train")
dataset

Let's look at the first row:

In [ ]:
dataset[0]["question"]

In [ ]:
dataset[0]["answer"]

We notice all answers like about have a ####, so we extract it:

In [ ]:
def extract_hash_answer(text):
    if "####" not in text: return None
    return text.split("####")[1].strip()
extract_hash_answer(dataset[0]["answer"])

We now create a system prompt which can be customized. We add 4 extra symbols for working out or thinking / reasoning sections and a final answer:

In [ ]:
reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

Let's map the dataset! and see the first row:

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["question"]},
    ],
    "answer": extract_hash_answer(x["answer"]),
})
dataset[0]

We create a regex format to match the reasoning sections and answers:

In [ ]:
import re

match_format = re.compile(
    rf"^[\s]{{0,}}"\
    rf"{reasoning_start}.+?{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

We verify it works:

In [ ]:
match_format.search(
    "<start_working_out>Let me think!<end_working_out>"\
    "<SOLUTION>2</SOLUTION>",
)

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(0)
            continue
        # Correct answer gets 3 points!
        if guess == true_answer:
            score += 3.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 1.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 1.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 0.5
                else: score -= 1.5 # Penalize wrong answers
            except:
                score -= 1.5 # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [ ]:
match_numbers = re.compile(
    solution_start + r".*?([\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print('*'*20, f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(0)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(1.5 if guess == true_answer else -0.5)
        except:
            scores.append(0)
            continue
    return scores

Get the maximum prompt length so we don't accidentally truncate it!

In [ ]:
max(dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
).map(lambda x: {"length" : len(x["tokens"])})["length"])

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = 287 + 1 # + 1 just in case!

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 500,
    save_steps = 250,
    max_grad_norm = 1.0,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role": "user", "content": "What is the sqrt of 101?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("advanced_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/advanced_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("advanced_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/advanced_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("advanced_lora")
    tokenizer.save_pretrained("advanced_lora")
if False:
    model.push_to_hub("HF_USERNAME/advanced_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/advanced_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("advanced_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/advanced_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("advanced_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/advanced_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("advanced_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/advanced_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/advanced_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `advanced_finetune.Q8_0.gguf` file or `advanced_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).